# 🧑‍🎓 PySpark Workbook: Star Wars API Integration

## 🎯 Obiettivo del laboratorio

* Recupera dati da un'API REST (SWAPI)

* Normalizza e struttura i dati in DataFrame

* Crea due tabelle: planets e films

* Costruisce una tabella relazionale planet_film_map

* Esegue join e query per esplorare le relazioni




## 📦 Step 1: Inizializza Spark
## ✅ Output atteso: SparkSession attiva.


In [4]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
#Check this site for the latest download link https://www.apache.org/dyn/closer.lua/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!wget -q https://dlcdn.apache.org/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark
!pip install pyspark
!pip install py4j

import os
import sys
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
# os.environ["SPARK_HOME"] = "/content/spark-3.2.1-bin-hadoop3.2"


import findspark
findspark.init()
findspark.find()

import pyspark

from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F

spark= SparkSession \
       .builder \
       .appName("Our First Spark Example") \
       .getOrCreate()

spark

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:3 https://cli.github.com/packages stable InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,125 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,441 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [5,976 kB]
Get:13 http://archive.ubuntu.com/ubuntu jam

## 🌍 Step 2: Recupera i dati dei pianeti
## ✅ Output atteso: DataFrame con colonne come name, climate, terrain, films.

In [6]:

import requests
import pandas as pd

planet_data = []
url = "https://swapi.dev/api/planets/"
while url:
    response = requests.get(url)
    data = response.json()
    planet_data.extend(data["results"])
    url = data.get("next")

planet_df = pd.json_normalize(planet_data)
spark_planets = spark.createDataFrame(planet_df)

spark_planets.show()

+--------------+---------------+--------------+--------+--------------------+--------------------+--------------------+-------------+-------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|          name|rotation_period|orbital_period|diameter|             climate|             gravity|             terrain|surface_water|   population|           residents|               films|             created|              edited|                 url|
+--------------+---------------+--------------+--------+--------------------+--------------------+--------------------+-------------+-------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|      Tatooine|             23|           304|   10465|                arid|          1 standard|              desert|            1|       200000|[https://swapi.de...|[https://swapi.de...|2014-12-09T13:50:...|2014-12-20T20:58:...|https://sw


## 🎬 Step 3: Recupera i dati dei film
## ✅ Output atteso: DataFrame con colonne come title, episode_id, release_date, url.


In [7]:
film_data = []
url = "https://swapi.dev/api/films/"
response = requests.get(url)
film_data.extend(response.json()["results"])

film_df = pd.json_normalize(film_data)
spark_films = spark.createDataFrame(film_df)

spark_films.show()


+--------------------+----------+--------------------+----------------+--------------------+------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|               title|episode_id|       opening_crawl|        director|            producer|release_date|          characters|             planets|           starships|            vehicles|             species|             created|              edited|                 url|
+--------------------+----------+--------------------+----------------+--------------------+------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|          A New Hope|         4|It is a period of...|    George Lucas|Gary Kurtz, Rick ...|  1977-05-25|[https://swapi.de...|[https://swapi.de...|[https://swapi.de...|[https://s


## 🔗 Step 4: Crea tabella relazionale pianeti-film
  
## ✅ Output atteso: Ogni riga collega un pianeta a un URL di film.


In [11]:
from pyspark.sql.functions import explode, col

planet_film_map = spark_planets.select(
    col("name").alias("planet_name"),
    explode("films").alias("film_url")
)

planet_film_map.count()
planet_film_map.show()



+-----------+--------------------+
|planet_name|            film_url|
+-----------+--------------------+
|   Tatooine|https://swapi.dev...|
|   Tatooine|https://swapi.dev...|
|   Tatooine|https://swapi.dev...|
|   Tatooine|https://swapi.dev...|
|   Tatooine|https://swapi.dev...|
|   Alderaan|https://swapi.dev...|
|   Alderaan|https://swapi.dev...|
|   Yavin IV|https://swapi.dev...|
|       Hoth|https://swapi.dev...|
|    Dagobah|https://swapi.dev...|
|    Dagobah|https://swapi.dev...|
|    Dagobah|https://swapi.dev...|
|     Bespin|https://swapi.dev...|
|      Endor|https://swapi.dev...|
|      Naboo|https://swapi.dev...|
|      Naboo|https://swapi.dev...|
|      Naboo|https://swapi.dev...|
|      Naboo|https://swapi.dev...|
|  Coruscant|https://swapi.dev...|
|  Coruscant|https://swapi.dev...|
+-----------+--------------------+
only showing top 20 rows



33






## 🔍 Step 5: Esegui il join con la tabella dei film
## ✅ Output atteso: Tabella finale con planet_name e title del film.


In [14]:
planet_film_joined = planet_film_map.join(
    spark_films.select(col("title"), col("url").alias("film_url")),
    on="film_url",
    how="inner"
)

planet_film_joined.select("planet_name", "title").show()


+-----------+--------------------+
|planet_name|               title|
+-----------+--------------------+
|   Tatooine|          A New Hope|
|   Alderaan|          A New Hope|
|   Yavin IV|          A New Hope|
|       Hoth|The Empire Strike...|
|    Dagobah|The Empire Strike...|
|     Bespin|The Empire Strike...|
|Ord Mantell|The Empire Strike...|
|   Tatooine|  Return of the Jedi|
|    Dagobah|  Return of the Jedi|
|      Endor|  Return of the Jedi|
|      Naboo|  Return of the Jedi|
|  Coruscant|  Return of the Jedi|
|   Tatooine|  The Phantom Menace|
|      Naboo|  The Phantom Menace|
|  Coruscant|  The Phantom Menace|
|   Tatooine|Attack of the Clones|
|      Naboo|Attack of the Clones|
|  Coruscant|Attack of the Clones|
|     Kamino|Attack of the Clones|
|   Geonosis|Attack of the Clones|
+-----------+--------------------+
only showing top 20 rows




## 🧪 Bonus: Esplora la tabella species
## ✅ Output atteso: DataFrame con name, classification, homeworld, films.

In [16]:
species_data = []
url = "https://swapi.dev/api/species/"
while url:
    response = requests.get(url)
    data = response.json()
    species_data.extend(data["results"])
    url = data.get("next")

species_df = pd.json_normalize(species_data)
spark_species = spark.createDataFrame(species_df)

spark_species.show()

+--------------+--------------+-----------+--------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|          name|classification|designation|average_height|         skin_colors|         hair_colors|          eye_colors|average_lifespan|           homeworld|      language|              people|               films|             created|              edited|                 url|
+--------------+--------------+-----------+--------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|         Human|        mammal|   sentient|           180|caucasian, black,...|blonde, brown, bl...|brown, blue, gree...|             120|https://swapi.dev...|G